In [ ]:
""" TRANFER LEARNING FINE TUNING TRAINING"""
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F
from torchvision import models
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR, ExponentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.TrivialAugmentWide(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= train_transform
)

test_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

train_size = int(0.8 * len(train_dataset))
test_size = len(train_dataset) - train_size
torch.manual_seed(42)
indices = torch.randperm(len(train_dataset))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

from torch.utils.data import Subset
training_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)


train_dataloader = DataLoader(training_data, batch_size=64, shuffle = True, num_workers=4,
pin_memory=True,
persistent_workers=True,
prefetch_factor=2,)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle = True, num_workers=4,
pin_memory=True,
persistent_workers=True,
prefetch_factor=2,)

model = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model.fc.in_features
# Here the size of each output sample is set to 2.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
model.fc = nn.Sequential(nn.Linear(num_ftrs, 512),
                         nn.BatchNorm1d(512),
                         nn.ReLU(),
                         nn.Linear(512, 128),
                         nn.BatchNorm1d(128),
                         nn.ReLU(),
                         nn.Linear(128,10))

torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler("cuda")

model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
learning_rate = (1e-3)
batch_size = 64
epochs = 60
# Observe that all parameters are being optimized
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-5)

# Decay LR by a factor of 0.1 every 7 epochs
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=(0))
x = torch.randn(2, 3, 224, 224).to(device)
print(model(x).shape)

max_correct = 0

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
          pred = model(X)
          loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")



def test_loop(dataloader, model, loss_fn):
    global max_correct
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, non_blocking = True)
            y = y.to(device, non_blocking = True)
            with torch.amp.autocast("cuda"):
              pred = model(X)
              loss = loss_fn(pred, y)
            test_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    scheduler.step()
    ''' if correct > max_correct:
        torch.save(model, 'Build1.pth')
        max_correct = correct
 '''

#optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
''' from torch.optim.lr_scheduler import (
    StepLR,
    MultiStepLR,
    ExponentialLR,
    CosineAnnealingLR,
    OneCycleLR,
    ReduceLROnPlateau
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-5
)'''
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
    print(str(optimizer.param_groups[0]["lr"])+"\n")
print("Done!")
print(max_correct)

torch.save(model, 'Build1.pth')


KeyboardInterrupt: 

In [ ]:
""" TRANSFER LEARNING FINE TUNING PREDICTION"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import csv
from PIL import Image
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    ),
])

test_data = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

test_dataloader = DataLoader(test_data, batch_size=100, shuffle = True)


loss_fn = nn.CrossEntropyLoss()
model = torch.load("Build 2 Week 3.pth", weights_only= False)
model = model.to(device)
model.eval()

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

for t in range(1):
    test_loop(test_dataloader, model, loss_fn)


t = []
for i in range(4000):
  t.append("img_"+"0"*(5-len(str(i)))+str(i)+".jpg")
with open("Build1 Predictions.csv", "w") as file:
    file.write("img_id,label\n")
for i in t:
  x = Path("drive/MyDrive/Images2/"+i)
  img = Image.open(x)
  img_tensor = test_transform(img)
  img_tensor = img_tensor.unsqueeze(0).to(device)

  with torch.no_grad():
    pred = model(img_tensor)

  prediction = pred.argmax(1).item()

  with open("Build1 Predictions.csv", "a") as file:
    file.write(i+","+str(prediction)+"\n")



print("Done!")




Test Error: 
 Accuracy: 99.8%, Avg loss: 0.009563 



KeyboardInterrupt: 

In [ ]:
""" TRANFER LEARNING FROZEN BACKBONE TRAINING"""
# Uncomment the following line, run this cell, and then restart your runtime (Runtime -> Restart runtime) to fix library incompatibility issues.
#!pip install torch==2.2.0 torchvision==0.17.0
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F
from torchvision import models
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.optim.lr_scheduler import ExponentialLR, CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.TrivialAugmentWide(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= train_transform
)

test_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

train_size = int(0.8 * len(train_dataset))
test_size = len(train_dataset) - train_size

indices = torch.randperm(len(train_dataset))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

from torch.utils.data import Subset
training_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)


train_dataloader = DataLoader(training_data, batch_size=64, shuffle = True, num_workers=4,
pin_memory=True,
persistent_workers=True,
prefetch_factor=2,)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle = True, num_workers=4,
pin_memory=True,
persistent_workers=True,
prefetch_factor=2,)

resnet = models.resnet18(weights='IMAGENET1K_V1')

# Keep everything except avgpool and fc
backbone = nn.Sequential(*list(resnet.children())[:-2])
for param in backbone.parameters():
    param.requires_grad = False

class ResidualBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(
            512, 1024,
            kernel_size=3,
            padding=1,
            )
        self.bn1 = nn.BatchNorm2d(1024)

        self.conv2 = nn.Conv2d(
            1024, 1024,
            kernel_size=3,
            padding=1,
        )
        self.bn2 = nn.BatchNorm2d(1024)

        self.shortcut = nn.Sequential(
            nn.Conv2d(
                512, 1024,
                kernel_size=1,
            ),
            nn.BatchNorm2d(1024)
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out
model = nn.Sequential(
    backbone,
    ResidualBlock(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.LazyLinear(128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
model = model.to(device)

learning_rate = (1e-2)
batch_size = 64
epochs = 80
# Observe that only parameters of final layer are being optimized as
# opposed to before.

# Decay LR by a factor of 0.1 every 7 epochs

torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler("cuda")

model = model.to(device)

loss_fn = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
    momentum=0.9,
    weight_decay=1e-4
)

# Decay LR by a factor of 0.1 every 7 epochs
scheduler = CosineAnnealingLR(
    optimizer,
    T_max = epochs,
    eta_min = 1e-7,

)
x = torch.randn(2, 3, 224, 224).to(device)
print(model(x).shape)

max_correct = 0

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    backbone.eval()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
          pred = model(X)
          loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")



def test_loop(dataloader, model, loss_fn):
    global max_correct
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, non_blocking = True)
            y = y.to(device, non_blocking = True)
            with torch.amp.autocast("cuda"):
              pred = model(X)
              loss = loss_fn(pred, y)
            test_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    scheduler.step()
    ''' if correct > max_correct:
        torch.save(model, 'Build1.pth')
        max_correct = correct
 '''

#optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
''' from torch.optim.lr_scheduler import (
    StepLR,
    MultiStepLR,
    ExponentialLR,
    CosineAnnealingLR,
    OneCycleLR,
    ReduceLROnPlateau
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-5
)'''
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
    print(str(optimizer.param_groups[0]["lr"])+"\n")
print("Done!")
print(max_correct)

torch.save(model, 'Build2.pth')


100%|██████████| 94.3M/94.3M [00:00<00:00, 360MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 157MB/s]


torch.Size([2, 10])
Epoch 1
-------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


loss: 2.340485  [   64/21600]
loss: 0.786266  [ 6464/21600]
loss: 0.348031  [12864/21600]
loss: 0.351630  [19264/21600]
Test Error: 
 Accuracy: 92.6%, Avg loss: 0.214357 

0.009996145219751803

Epoch 2
-------------------------------
loss: 0.168499  [   64/21600]
loss: 0.419433  [ 6464/21600]
loss: 0.421569  [12864/21600]
loss: 0.324410  [19264/21600]
Test Error: 
 Accuracy: 93.6%, Avg loss: 0.193111 

0.009984586822798954

Epoch 3
-------------------------------
loss: 0.283846  [   64/21600]
loss: 0.236954  [ 6464/21600]
loss: 0.293518  [12864/21600]
loss: 0.309271  [19264/21600]
Test Error: 
 Accuracy: 93.9%, Avg loss: 0.177486 

0.009965342631351785

Epoch 4
-------------------------------
loss: 0.341435  [   64/21600]
loss: 0.168541  [ 6464/21600]
loss: 0.237759  [12864/21600]
loss: 0.294903  [19264/21600]
Test Error: 
 Accuracy: 94.3%, Avg loss: 0.172586 

0.009938442318558659

Epoch 5
-------------------------------
loss: 0.269328  [   64/21600]
loss: 0.315565  [ 6464/21600]
loss

In [ ]:
""" TRANSFER LEARNING FROZEN BACKBONE PREDICTION"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import csv
from PIL import Image
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F

class ResidualBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(
            512, 1024,
            kernel_size=3,
            padding=1,
            )
        self.bn1 = nn.BatchNorm2d(1024)

        self.conv2 = nn.Conv2d(
            1024, 1024,
            kernel_size=3,
            padding=1,
        )
        self.bn2 = nn.BatchNorm2d(1024)

        self.shortcut = nn.Sequential(
            nn.Conv2d(
                512, 1024,
                kernel_size=1,
            ),
            nn.BatchNorm2d(1024)
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    ),
])

test_data = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

test_dataloader = DataLoader(test_data, batch_size=100, shuffle = True)


loss_fn = nn.CrossEntropyLoss()
model = torch.load("Build 2 Week 3.pth", weights_only= False)
model = model.to(device)
model.eval()

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

for t in range(1):
    test_loop(test_dataloader, model, loss_fn)


t = []
for i in range(4000):
  t.append("img_"+"0"*(5-len(str(i)))+str(i)+".jpg")
with open("Build1 Predictions.csv", "w") as file:
    file.write("img_id,label\n")
for i in t:
  x = Path("drive/MyDrive/Images2/"+i)
  img = Image.open(x)
  img_tensor = test_transform(img)
  img_tensor = img_tensor.unsqueeze(0).to(device)

  with torch.no_grad():
    pred = model(img_tensor)

  prediction = pred.argmax(1).item()

  with open("Build1 Predictions.csv", "a") as file:
    file.write(i+","+str(prediction)+"\n")



print("Done!")


